# 01. 데이터 구조와 품질 확인
SKALA 3반 · U095 · 이연주 | DAY 1 | 2026-10-01

세 개 `.mat` 파일을 직접 읽어 **배터리 셀 단위**로 정리한다. 원본 데이터는 수정하지 않는다.
Scratch의 `batch → cell → summary / cycles` 구조를 그대로 사용하되, h5py로 필요한 배열만 읽는다.
이는 mat73의 미지원 문자열 경고와 전체 시계열 로딩 비용을 피하기 위한 구현 선택이다.

**실행:** 프로젝트의 `.venv` 커널 선택 → Restart Kernel → Run All.
이 노트북은 모델을 학습하지 않는다. CSV는 UTF-8 BOM 형식이다.
셀 ID는 파일 내 0부터 시작하는 인덱스이며, 예를 들어 `b1c0`은 Batch 1의 첫 셀이다.

In [7]:
from pathlib import Path
import sys, os
ROOT = Path(os.environ.get("ESS_PROJECT_ROOT", Path.cwd())).resolve()
if ROOT.name == "notebooks": ROOT = ROOT.parent
assert (ROOT / "data").exists(), "프로젝트 루트에서 실행한다."
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
import day1_analysis as da
import day1_review as review
da.configure()
OUT = ROOT / "results" / "day1"
(OUT / "tables").mkdir(parents=True, exist_ok=True)
(OUT / "figures").mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 90)
print("프로젝트:", ROOT)
print("random_state =", da.RANDOM_STATE)

프로젝트: /Users/yeonju/workspace/데이터 분석 mini-Project/ess-battery-project
random_state = 42


## 1. 데이터 출처와 구조
Batch 1 = 2017-05-12, Batch 2 = 2018-02-20, Batch 3 = 2018-04-12.

`summary`: 사이클별 QD·QC·IR·온도·충전시간·실제 cycle 번호.
`cycles`: 사이클 내부의 전압·전류·Qdlin 등. `Vdlin`: Qdlin의 공통 전압축.
`cycle_life`: 제공 파일의 정답 필드. 누락되거나 종료가 불명확하면 임의로 보충하지 않는다.

In [8]:
cells, manifest = da.load_batches(ROOT / "data")
features, quality = da.feature_table(cells)
summary, corr = da.make_tables(cells, features, quality, manifest, OUT)
display(summary.round(3))

,batch,cells,labels,missing_labels,cycles,policies,min,q25,median,mean,q75,max,std,short_n,long_n,short_pct,long_pct,below550,above_eq550,valid_dq,eligible
0,Batch 1,46,46,0,38811,23,534.0,703.25,858.5,844.717,914.25,1227.0,184.629,0,10,0.000,21.739,1,45,46,36
1,Batch 2,47,39,8,26904,20,392.0,439.50,472.0,565.744,508.50,1186.0,222.202,28,3,71.795,7.692,30,9,47,39
2,Batch 3,46,44,2,51007,8,541.0,828.00,1005.5,1059.659,1155.25,1935.0,313.870,0,23,0.000,52.273,1,43,46,40


In [9]:
display(manifest)
print("첫 셀 summary 자료형")
display(cells[0]["summary"].dtypes.to_frame("dtype"))
display(cells[0]["summary"].head())
for b in da.FILES:
    c = next(c for c in cells if c["batch"] == b)
    print(b, "Vdlin:", len(c["voltage"]), "전압 범위:", c["voltage"].min(), c["voltage"].max())
    print("Qdlin cycle 10/100:", c["qcurves"][10].shape, c["qcurves"][100].shape)

,batch,file,bytes,cell_count,fields
0,Batch 1,2017-05-12_batchdata_updated_struct_errorcorrect.mat,3025320241,46,"Vdlin,barcode,channel_id,cycle_life,cycles,policy,policy_readable,summary"
1,Batch 2,2018-02-20_batchdata_updated_struct_errorcorrect.mat,2022599329,47,"Vdlin,barcode,channel_id,cycle_life,cycles,policy,policy_readable,summary"
2,Batch 3,2018-04-12_batchdata_updated_struct_errorcorrect.mat,3236690412,46,"Vdlin,barcode,channel_id,cycle_life,cycles,policy,policy_readable,summary"


첫 셀 summary 자료형


,dtype
QDischarge,float64
QCharge,float64
IR,float64
Tavg,float64
Tmax,float64
Tmin,float64
chargetime,float64
cycle,float64


,QDischarge,QCharge,IR,Tavg,Tmax,Tmin,chargetime,cycle
0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.0
1,1.070689,1.071042,0.016742,31.875011,35.652016,29.566130,13.341250,2.0
2,1.071900,1.071674,0.016724,31.931490,35.692978,29.604385,13.425777,3.0
3,1.072510,1.072304,0.016681,31.932603,35.680588,29.744202,13.425167,4.0
4,1.073174,1.072970,0.016662,31.959322,35.728691,29.644709,13.341442,5.0


Batch 1 Vdlin: 1000 전압 범위: 2.0 3.5
Qdlin cycle 10/100: (1000,) (1000,)
Batch 2 Vdlin: 1000 전압 범위: 2.0 3.5
Qdlin cycle 10/100: (1000,) (1000,)
Batch 3 Vdlin: 1000 전압 범위: 2.0 3.5
Qdlin cycle 10/100: (1000,) (1000,)


## 2. 결측과 0은 다르다
summary의 NaN 수가 0이어도 첫 사이클의 0, 과도한 충전시간, 비정상 용량은 남아 있다.
품질 규칙은 전체 분포를 보고 최적화한 임계값이 아닌, 명시적인 탐색 규칙이다.

- QD 시각화/초기 기울기: `0 < QD ≤ 1.5 Ah`만 사용. EOL 이하를 삭제하는 필터가 아니다.
- 온도·IR: 0 이하를 결측 후보로 처리. 충전시간: 0 이하 또는 60분 초과는 중앙값 계산에서 제외.
- 초기 요약: 원래 사이클 번호 2~100. 첫 사이클은 Batch 간 초기화 차이 때문에 제외.
- Qdlin: 10·100사이클이 각각 1,000개 유한값인지 검증. 배열 위치를 사이클 번호로 착각하지 않는다.
- 이런 규칙의 민감도는 DAY 2 내부 검증에서 확인해야 한다.

In [10]:
display(quality.groupby(["batch", "field"])[["missing", "zero", "qd_outside_0_1_5", "charge_over60"]].sum())
display(features.groupby("batch")[["has_cycle5", "has_cycle100", "dq_valid"]].sum())

missing  zero  qd_outside_0_1_5  charge_over60
batch   field                                                     
Batch 1 IR                0    65                 0              0
        QCharge           0    46                 0              0
        QDischarge        0    46                48              0
        Tavg              0    46                 0              0
        Tmax              0    46                 0              0
        Tmin              0    46                 0              0
        chargetime        0    46                 0              6
Batch 2 IR                0  4904                 0              0
        QCharge           0     0                 0              0
        QDischarge        0     0                11              0
        Tavg              0     0                 0              0
        Tmax              0     0                 0              0
        Tmin              0    24                 0              0
        chargetime        0     0                 0             22
Batch 3 IR                0     0                 0              0
        QCharge           0     0                 0              0
        QDischarge        0     0                 0              0
        Tavg              0     0                 0              0
        Tmax              0     0                 0              0
        Tmin              0     0                 0              0
        chargetime        0     0                 0              0

,has_cycle5,has_cycle100,dq_valid
batch,,,
Batch 1,46,46,46
Batch 2,47,47,47
Batch 3,46,46,46


## 3. 정답 신뢰도와 분석 집합
전체 139개 셀을 원본 구조/곡선에 포함한다. 수명 분포는 유한한 정답 129개 기준이다.
상관관계와 모델 후보 분석은 다음 사유를 표시한 뒤 115개를 사용한다.

- Batch 1: `0~4`는 공개 코드상 다음 배치로 이어진 셀이고 현재 종료 QD도 높다. 필요한 2017-06-30 파일이 없으므로 수명 보정 대신 보류한다.
- Batch 1: `8, 10, 12, 13, 22`는 논문 공개 코드의 종료 미확인 셀이다. 현재 종료 QD도 0.91~0.97 Ah로 남아 있다.
- Batch 2: 정답 NaN 8개는 특수 프로토콜이며 지도학습/수명 상관분석에서 제외한다. 10/100사이클 배열 자체는 존재한다.
- Batch 3: 공개 코드의 품질 경고 셀 `2, 23, 32, 37, 42, 43`은 주분석에서 제외한다. 이 중 23, 32는 정답도 NaN이다.

이 제외는 '짧게 산 셀을 지우는 것'이 아니다. 실제로 짧은 유효 셀은 유지한다.
모든 셀을 사용한 상관관계도 함께 저장하여 제외에 따른 변화를 확인한다.

In [11]:
audit = features[["cell_id", "batch", "cycle_life", "last_qd", "dq_valid", "analysis_eligible", "review_reason"]]
display(audit[~audit.analysis_eligible])
print("주분석 셀 수")
display(features.groupby("batch").analysis_eligible.sum())

,cell_id,batch,cycle_life,last_qd,dq_valid,analysis_eligible,review_reason
0,b1c0,Batch 1,1190.0,1.026210,True,False,unresolved_EOl_or_continuation
1,b1c1,Batch 1,1179.0,1.038225,True,False,unresolved_EOl_or_continuation
2,b1c2,Batch 1,1177.0,1.043279,True,False,unresolved_EOl_or_continuation
3,b1c3,Batch 1,1226.0,0.970921,True,False,unresolved_EOl_or_continuation
4,b1c4,Batch 1,1227.0,1.021960,True,False,unresolved_EOl_or_continuation
8,b1c8,Batch 1,879.0,0.969025,True,False,unresolved_EOl_or_continuation
10,b1c10,Batch 1,906.0,0.961019,True,False,unresolved_EOl_or_continuation
12,b1c12,Batch 1,902.0,0.913119,True,False,unresolved_EOl_or_continuation
13,b1c13,Batch 1,897.0,0.923136,True,False,unresolved_EOl_or_continuation
22,b1c22,Batch 1,891.0,0.963279,True,False,unresolved_EOl_or_continuation


주분석 셀 수


batch
Batch 1    36
Batch 2    39
Batch 3    40
Name: analysis_eligible, dtype: int64

## 4. 재현성 점검
파일 전체 SHA-256과 라이브러리 버전을 저장한다. 데이터 파일명만 같고 내용이 달라지는 경우를 구분할 수 있다.
아래 검사는 실제 배열 크기, 사이클 일치, 셀 수, 피처 유한값을 확인한다.

In [12]:
import hashlib, json
fingerprints = []
for _, row in manifest.iterrows():
    path = ROOT / "data" / row["file"]
    sha = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            sha.update(chunk)
    fingerprints.append({"file": row["file"], "sha256": sha.hexdigest(), "bytes": path.stat().st_size})
pd.DataFrame(fingerprints).to_csv(OUT / "tables" / "data_fingerprints.csv", index=False)
display(pd.DataFrame(fingerprints))
assert len(cells) == 139 and features.cell_id.is_unique
assert summary.cells.tolist() == [46, 47, 46]
assert summary.missing_labels.tolist() == [0, 8, 2]
assert summary.eligible.tolist() == [36, 39, 40]
assert features.dq_valid.all()
assert np.isfinite(features.loc[features.analysis_eligible, "log_dq_var"]).all()
print("PASS: 139개 셀, 129개 유효 정답, 115개 주분석 셀, 모든 ΔQ 배열 검증 완료")
display(json.loads((OUT / "environment.json").read_text()))

,file,sha256,bytes
0,2017-05-12_batchdata_updated_struct_errorcorrect.mat,9d928ab978f0e3c70b31cb833a749fedd35094d01af76475d69b40aa3497f5ba,3025320241
1,2018-02-20_batchdata_updated_struct_errorcorrect.mat,3487db505ad08b6c805f41dd0560e73b54077c76aca04f706f06e2356ccf7845,2022599329
2,2018-04-12_batchdata_updated_struct_errorcorrect.mat,62c30e413b63e6144720e016deed3661fac8468641794a5807b123fe84717998,3236690412


PASS: 139개 셀, 129개 유효 정답, 115개 주분석 셀, 모든 ΔQ 배열 검증 완료


{'python': '3.11.15',
 'platform': 'macOS-26.6.2-arm64-arm-64bit',
 'random_state': 42,
 'libraries': {'numpy': '2.4.6',
  'pandas': '3.0.6',
  'h5py': '3.16.0',
  'scipy': '1.17.1',
  'matplotlib': '3.11.2',
  'scikit-learn': '1.9.1',
  'nbformat': '5.11.1',
  'nbclient': '0.11.0'}}

## 확인 결과를 어떻게 이해하는 방법
Scratch의 38,811행은 **38,811개의 독립적인 배터리**가 아니다. Batch 1 배터리는 46개이며 주분석은 36개이다.
따라서 모델은 셀당 1행으로 학습해야 하며, 무작위로 사이클 행을 나누면 안 된다.
다음 `02_day1_eda.ipynb`에서 다섯 질문의 그래프·해석·모델 전략을 확인한다.

출처: [논문](https://doi.org/10.1038/s41560-019-0356-8),
[저자 공개 로딩 코드](https://github.com/rdbraatz/data-driven-prediction-of-battery-cycle-life-before-capacity-degradation/blob/master/Load%20Data.ipynb).